# CDC PLACES Census Tract Data Download - 2024

This notebook downloads census tract-level health indicators from the CDC PLACES dataset for any U.S. state or territory.

---

## About CDC PLACES

The **CDC PLACES** (Population Level Analysis and Community Estimates) dataset provides model-based estimates for chronic disease risk factors, health outcomes, and clinical preventive service use for all U.S. counties, census tracts, and places.

**Data includes:**
- Chronic conditions (diabetes, hypertension, COPD, asthma, etc.)
- Health behaviors (smoking, obesity, physical inactivity)
- Preventive services (health checkups, cancer screenings)
- Mental health indicators
- Access to care measures

---

## Instructions

1. Run all cells in order (Runtime → Run all)
2. Select your state from the dropdown menu
3. Wait for the data to download (progress will be displayed)
4. Review the data summary and visualizations
5. Download the CSV file to your computer

---

## 1. Setup and Installation

Install required Python packages (this may take a minute).

In [ ]:
# Install required packages
!pip install -q requests pandas matplotlib seaborn

print("✓ All packages installed successfully!")

## 2. Import Libraries

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from IPython.display import display, HTML, FileLink
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Set plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully!")

## 3. State Selection Configuration

In [ ]:
# State mapping dictionary
STATE_MAPPING = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR',
    'California': 'CA', 'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE',
    'Florida': 'FL', 'Georgia': 'GA', 'Hawaii': 'HI', 'Idaho': 'ID',
    'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA', 'Kansas': 'KS',
    'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
    'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS',
    'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV',
    'New Hampshire': 'NH', 'New Jersey': 'NJ', 'New Mexico': 'NM', 'New York': 'NY',
    'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK',
    'Oregon': 'OR', 'Pennsylvania': 'PA', 'Rhode Island': 'RI', 'South Carolina': 'SC',
    'South Dakota': 'SD', 'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT',
    'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA', 'West Virginia': 'WV',
    'Wisconsin': 'WI', 'Wyoming': 'WY',
    'District of Columbia': 'DC', 'Puerto Rico': 'PR'
}

# Reverse mapping
ABBR_TO_STATE = {v: k for k, v in STATE_MAPPING.items()}

print("✓ State configuration loaded!")
print(f"\nTotal states/territories available: {len(STATE_MAPPING)}")

## 4. Data Download Functions

In [ ]:
def download_cdc_places_data(state_abbr):
    """
    Download CDC PLACES census tract data for a given state.
    
    Parameters:
    -----------
    state_abbr : str
        Two-letter state abbreviation (e.g., 'MA', 'CA', 'NY')
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame containing the census tract health data
    """
    
    print(f"\n{'='*70}")
    print(f"  Downloading data for {ABBR_TO_STATE.get(state_abbr, state_abbr)} ({state_abbr})")
    print(f"{'='*70}\n")
    
    # CDC PLACES API endpoint
    base_url = "https://data.cdc.gov/resource/cwsq-ngmh.json"
    
    # Initialize variables
    all_data = []
    offset = 0
    limit = 50000  # Maximum allowed by Socrata API
    
    # Fetch data with pagination
    while True:
        print(f"📥 Fetching records {offset + 1:,} to {offset + limit:,}...", end=" ")
        
        # Build query parameters
        params = {
            '$where': f"stateabbr='{state_abbr}'",
            '$limit': limit,
            '$offset': offset,
            '$order': ':id'
        }
        
        try:
            # Make API request
            response = requests.get(base_url, params=params, timeout=120)
            response.raise_for_status()
            
            # Parse JSON
            batch_data = response.json()
            
            # Check if we got any data
            if not batch_data:
                print("✓ No more data")
                break
            
            # Add to collection
            all_data.extend(batch_data)
            print(f"✓ Retrieved {len(batch_data):,} records (Total: {len(all_data):,})")
            
            # If we got fewer records than the limit, we're done
            if len(batch_data) < limit:
                break
            
            # Move to next batch
            offset += limit
            
            # Rate limiting - be nice to the API
            time.sleep(1)
            
        except requests.exceptions.RequestException as e:
            print(f"\n❌ Error: {e}")
            print("\nNote: If you see a 404 error, the dataset ID may have been updated.")
            print("Check: https://data.cdc.gov/browse?category=500+Cities+%26+Places")
            return None
    
    # Convert to DataFrame
    if not all_data:
        print(f"\n❌ No data found for {state_abbr}")
        return None
    
    df = pd.DataFrame(all_data)
    
    print(f"\n{'='*70}")
    print(f"  ✓ Download Complete!")
    print(f"{'='*70}")
    print(f"\nTotal records retrieved: {len(df):,}")
    print(f"Unique census tracts: {df['locationname'].nunique():,}")
    print(f"Health measures included: {df['measureid'].nunique():,}")
    
    return df


def save_data(df, state_abbr):
    """
    Save DataFrame to CSV file.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame to save
    state_abbr : str
        State abbreviation for filename
    
    Returns:
    --------
    str
        Filename of saved CSV
    """
    filename = f"cdc_places_census_tract_{state_abbr}_2024.csv"
    df.to_csv(filename, index=False)
    print(f"\n💾 Data saved to: {filename}")
    print(f"📊 File size: {pd.io.common.get_filepath_or_buffer(filename)[0].__sizeof__() / 1024**2:.2f} MB")
    return filename


print("✓ Functions loaded successfully!")

## 5. Select State and Download Data

Choose your state from the dropdown below and click **"Download Data"**.

In [ ]:
# Create dropdown widget
state_dropdown = widgets.Dropdown(
    options=sorted(STATE_MAPPING.keys()),
    value='Massachusetts',
    description='State:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

# Create download button
download_button = widgets.Button(
    description='Download Data',
    button_style='primary',
    tooltip='Click to download CDC PLACES data',
    icon='download',
    layout=widgets.Layout(width='200px', height='40px')
)

# Output area
output = widgets.Output()

# Global variable to store data
global_data = {'df': None, 'state_abbr': None, 'filename': None}

def on_download_button_clicked(b):
    """Handle download button click."""
    with output:
        output.clear_output()
        
        # Get selected state
        state_name = state_dropdown.value
        state_abbr = STATE_MAPPING[state_name]
        
        # Download data
        df = download_cdc_places_data(state_abbr)
        
        if df is not None:
            # Save data
            filename = save_data(df, state_abbr)
            
            # Store in global variable for later use
            global_data['df'] = df
            global_data['state_abbr'] = state_abbr
            global_data['filename'] = filename
            
            print("\n✅ Ready for analysis! Run the next cells to explore the data.")
        else:
            print("\n❌ Download failed. Please try again.")

# Attach click handler
download_button.on_click(on_download_button_clicked)

# Display widgets
display(widgets.VBox([
    widgets.HBox([state_dropdown, download_button]),
    output
]))

## 6. Data Preview

Preview the first few rows of downloaded data.

In [ ]:
if global_data['df'] is not None:
    df = global_data['df']
    
    print("\n📋 DATA PREVIEW")
    print("=" * 70)
    
    # Display shape
    print(f"\nDataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    
    # Display column names
    print(f"\nColumns ({len(df.columns)}):")
    print(", ".join(df.columns.tolist()))
    
    # Display first few rows
    print("\n📊 First 5 rows:")
    display(df.head())
    
    # Display data types
    print("\n🔍 Data types:")
    display(df.dtypes)
    
else:
    print("⚠️ No data available. Please download data first using the button above.")

## 7. Available Health Measures

View all health indicators included in the dataset.

In [ ]:
if global_data['df'] is not None:
    df = global_data['df']
    
    print("\n🏥 AVAILABLE HEALTH MEASURES")
    print("=" * 70)
    
    # Get unique measures
    measures = df[['measureid', 'measure', 'category']].drop_duplicates().sort_values('measureid')
    
    print(f"\nTotal measures: {len(measures)}\n")
    
    # Group by category
    if 'category' in measures.columns:
        for category in sorted(measures['category'].unique()):
            cat_measures = measures[measures['category'] == category]
            print(f"\n{category}:")
            print("-" * 70)
            for _, row in cat_measures.iterrows():
                print(f"  • {row['measureid']:20s} - {row['measure']}")
    else:
        display(measures)
    
else:
    print("⚠️ No data available. Please download data first.")

## 8. Data Summary Statistics

Summary statistics for selected health measures.

In [ ]:
if global_data['df'] is not None:
    df = global_data['df']
    
    print("\n📈 SUMMARY STATISTICS")
    print("=" * 70)
    
    # Convert data_value to numeric if needed
    if 'data_value' in df.columns:
        df['data_value_numeric'] = pd.to_numeric(df['data_value'], errors='coerce')
    
    # Summary by measure
    if 'data_value_numeric' in df.columns and 'measure' in df.columns:
        summary = df.groupby('measure')['data_value_numeric'].agg([
            ('count', 'count'),
            ('mean', 'mean'),
            ('std', 'std'),
            ('min', 'min'),
            ('25%', lambda x: x.quantile(0.25)),
            ('median', 'median'),
            ('75%', lambda x: x.quantile(0.75)),
            ('max', 'max')
        ]).round(2)
        
        print("\nSummary by Health Measure:")
        display(summary)
    
    # Census tract coverage
    if 'locationname' in df.columns:
        print(f"\n📍 Geographic Coverage:")
        print(f"  Total census tracts: {df['locationname'].nunique():,}")
    
    # Data completeness
    print("\n📊 Data Completeness:")
    print(f"  Missing values by column:")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_df = pd.DataFrame({
        'Missing Count': missing,
        'Percentage': missing_pct
    })
    display(missing_df[missing_df['Missing Count'] > 0])
    
else:
    print("⚠️ No data available. Please download data first.")

## 9. Data Visualizations

Visualize health measure distributions across census tracts.

In [ ]:
if global_data['df'] is not None:
    df = global_data['df']
    
    # Convert data_value to numeric
    if 'data_value' in df.columns:
        df['data_value_numeric'] = pd.to_numeric(df['data_value'], errors='coerce')
    
    if 'data_value_numeric' in df.columns and 'measureid' in df.columns:
        # Get top 10 most common measures
        top_measures = df['measureid'].value_counts().head(10).index.tolist()
        
        # Plot distribution for top measures
        fig, axes = plt.subplots(5, 2, figsize=(15, 20))
        axes = axes.flatten()
        
        for i, measure_id in enumerate(top_measures):
            measure_data = df[df['measureid'] == measure_id]['data_value_numeric'].dropna()
            measure_name = df[df['measureid'] == measure_id]['measure'].iloc[0]
            
            axes[i].hist(measure_data, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
            axes[i].set_title(f"{measure_id}\n{measure_name}", fontsize=10, fontweight='bold')
            axes[i].set_xlabel('Value (%)', fontsize=9)
            axes[i].set_ylabel('Frequency', fontsize=9)
            axes[i].axvline(measure_data.mean(), color='red', linestyle='--', 
                           linewidth=2, label=f'Mean: {measure_data.mean():.1f}%')
            axes[i].legend(fontsize=8)
            axes[i].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.suptitle(f'Distribution of Health Measures - {ABBR_TO_STATE[global_data["state_abbr"]]}', 
                     fontsize=14, fontweight='bold', y=1.001)
        plt.show()
        
        # Create a correlation heatmap (if multiple measures)
        print("\n🗺️ Creating comparison chart...\n")
        
        # Pivot data for comparison
        pivot_df = df.pivot_table(
            index='locationname',
            columns='measureid',
            values='data_value_numeric',
            aggfunc='first'
        )
        
        if len(pivot_df.columns) > 1:
            # Calculate correlation matrix for top 15 measures
            corr_measures = pivot_df.columns[:15]
            corr_matrix = pivot_df[corr_measures].corr()
            
            plt.figure(figsize=(12, 10))
            sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
                       center=0, square=True, linewidths=1,
                       cbar_kws={"shrink": 0.8})
            plt.title(f'Correlation between Health Measures - {ABBR_TO_STATE[global_data["state_abbr"]]}',
                     fontsize=14, fontweight='bold', pad=20)
            plt.tight_layout()
            plt.show()
    else:
        print("⚠️ Unable to create visualizations - missing required columns.")
else:
    print("⚠️ No data available. Please download data first.")

## 10. Download CSV File

Download the data to your local computer.

In [ ]:
if global_data['filename'] is not None:
    from google.colab import files
    
    print("\n💾 DOWNLOAD CSV FILE")
    print("=" * 70)
    print(f"\nFilename: {global_data['filename']}")
    print(f"State: {ABBR_TO_STATE[global_data['state_abbr']]} ({global_data['state_abbr']})")
    print(f"Records: {len(global_data['df']):,}")
    print("\nClick the download link below to save the file to your computer.\n")
    
    # Trigger download
    files.download(global_data['filename'])
    
    print("\n✅ Download initiated! Check your browser's download folder.")
else:
    print("⚠️ No data available. Please download data first using the button in cell 5.")

## 11. Next Steps and Integration

### Integration with Census Data

You can merge this CDC PLACES data with census demographic data using the census tract GEOID:

```python
# Example: Merge with census data
import pandas as pd

# Load census data
census_data = pd.read_csv('census_data.csv')

# Pivot PLACES data to wide format
places_wide = df.pivot_table(
    index='locationname',
    columns='measureid',
    values='data_value',
    aggfunc='first'
).reset_index()

# Extract GEOID from locationname
places_wide['geoid'] = places_wide['locationname'].str.extract(r'Census Tract ([0-9.]+)')

# Merge datasets
combined = census_data.merge(places_wide, on='geoid', how='left')
```

### Analysis Ideas

1. **Spatial Analysis**: Map health outcomes by census tract
2. **Correlation Studies**: Examine relationships between health measures
3. **Demographic Patterns**: Link health data with census demographics
4. **Time Trends**: Compare with previous years' data
5. **Health Disparities**: Identify areas with higher disease burden

### Additional Resources

- **CDC PLACES Homepage**: https://www.cdc.gov/places
- **Data Dictionary**: https://www.cdc.gov/places/measure-definitions/index.html
- **Methodology**: https://www.cdc.gov/places/methodology/index.html
- **API Documentation**: https://dev.socrata.com/foundry/data.cdc.gov/cwsq-ngmh

---

## 📝 Citation

If you use CDC PLACES data in your research, please cite:

> Centers for Disease Control and Prevention. PLACES: Local Data for Better Health, Census Tract Data 2024 release. Accessed [date]. Available at: https://www.cdc.gov/places

---

**Notebook Version**: 1.0  
**Last Updated**: November 4, 2024  
**Project**: DNA methylation & adversity (R01MD014304-04)
